# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip install duckdb huggingface_hub

In [4]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token Loaded Successfully!")

Token Loaded Successfully!


In [5]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("Connected!")

Connected!


In [6]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

In [7]:
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [8]:
print(TABLES)

{'dim_clients': "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')", 'dim_content': "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')", 'fact_daily': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')", 'fact_daily_sample': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')", 'fact_query_90d': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet')"}


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule:
I will prioritize content that has low impressions, low CTR, and a high average position because these pages are more likely to benefit from optimization.

Reason Codes:
- LOW_CTR
- LOW_IMPRESSIONS
- HIGH_POSITION

Action:
Optimize content for SEO and improve titles and descriptions.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM {TABLES['fact_daily']}
WHERE DATE_TRUNC('month', report_date) = DATE '2026-03-01'
LIMIT 1000
""").df()

df.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Ranking Rule:

Pages with low CTR and poor average position receive a higher score because they are better candidates for SEO optimization.

Action Label:
Optimize SEO

Reason Code:
LOW_CTR_HIGH_POSITION

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import os

# Create a copy
queue = df.copy()

# Calculate CTR (%)
queue["ctr"] = (
    queue["gsc_clicks"] /
    queue["gsc_impressions"].replace(0, 1)
) * 100

# Baseline Score
queue["baseline_score"] = (
    (100 - queue["ctr"]) +
    (queue["gsc_avg_position"] * 2)
)

# Reason Code
queue["reason_code"] = "LOW_CTR_HIGH_POSITION"

# Action Label
queue["action"] = "Optimize SEO"

# Sort by score
queue = queue.sort_values(
    by="baseline_score",
    ascending=False
)

# Create output folder
os.makedirs("work/outputs", exist_ok=True)

# Save CSV
queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV Saved Successfully!")

queue.head(20)


CSV Saved Successfully!


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,baseline_score,reason_code,action
357,client_73cda7b4e4f265ea,content_17624ba0605baddc,2026-03-01,1,0,99.000000,0.0,298.000000,LOW_CTR_HIGH_POSITION,Optimize SEO
528,client_73cda7b4e4f265ea,content_0b4f8bb8e5510f9b,2026-03-01,1,0,98.000000,0.0,296.000000,LOW_CTR_HIGH_POSITION,Optimize SEO
94,client_73cda7b4e4f265ea,content_0f040e1b2668c08a,2026-03-01,2,0,94.000000,0.0,288.000000,LOW_CTR_HIGH_POSITION,Optimize SEO
875,client_73cda7b4e4f265ea,content_ae889f6148cc06db,2026-03-01,1,0,94.000000,0.0,288.000000,LOW_CTR_HIGH_POSITION,Optimize SEO
850,client_73cda7b4e4f265ea,content_b26d43d9c0ce0198,2026-03-01,1,0,94.000000,0.0,288.000000,LOW_CTR_HIGH_POSITION,Optimize SEO
369,client_73cda7b4e4f265ea,content_04143616bdd1228e,2026-03-01,3,0,92.333333,0.0,284.666667,LOW_CTR_HIGH_POSITION,Optimize SEO
648,client_73cda7b4e4f265ea,content_28de717dd8841f0a,2026-03-01,3,0,91.666667,0.0,283.333333,LOW_CTR_HIGH_POSITION,Optimize SEO
219,client_73cda7b4e4f265ea,content_f640837a5b04b7ee,2026-03-01,1,0,91.000000,0.0,282.000000,LOW_CTR_HIGH_POSITION,Optimize SEO
429,client_73cda7b4e4f265ea,content_de1ea1aae4988a57,2026-03-01,1,0,91.000000,0.0,282.000000,LOW_CTR_HIGH_POSITION,Optimize SEO
768,client_73cda7b4e4f265ea,content_d2904ae257bae57f,2026-03-01,1,0,90.000000,0.0,280.000000,LOW_CTR_HIGH_POSITION,Optimize SEO


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 Review

I reviewed the top 20 ranked pages generated by my baseline score.

Pages with low CTR and high average position were ranked higher because they have a higher chance of benefiting from SEO improvements.

Reason Code:
LOW_CTR_HIGH_POSITION

Action:
Optimize SEO

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20)

for i, row in top20.iterrows():
    print(f"{i+1}.")
    print(f"Action: {row['action']}")
    print(f"Reason Code: {row['reason_code']}")
    print("Confidence: Medium")
    print("What would make it wrong: The page may already be optimized, or low traffic may be caused by seasonal demand instead of SEO issues.")
    print("-" * 60)


358.
Action: Optimize SEO
Reason Code: LOW_CTR_HIGH_POSITION
Confidence: Medium
What would make it wrong: The page may already be optimized, or low traffic may be caused by seasonal demand instead of SEO issues.
------------------------------------------------------------
529.
Action: Optimize SEO
Reason Code: LOW_CTR_HIGH_POSITION
Confidence: Medium
What would make it wrong: The page may already be optimized, or low traffic may be caused by seasonal demand instead of SEO issues.
------------------------------------------------------------
95.
Action: Optimize SEO
Reason Code: LOW_CTR_HIGH_POSITION
Confidence: Medium
What would make it wrong: The page may already be optimized, or low traffic may be caused by seasonal demand instead of SEO issues.
------------------------------------------------------------
876.
Action: Optimize SEO
Reason Code: LOW_CTR_HIGH_POSITION
Confidence: Medium
What would make it wrong: The page may already be optimized, or low traffic may be caused by seasonal 

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks + Leakage Check

Some pages may receive a high score because they have naturally low traffic or seasonal demand.

No future information or label-derived features were used.

The baseline score only uses current observations such as impressions, clicks, CTR, and average position.

Therefore, no data leakage was introduced into the ranking process.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Weak Picks + Leakage Check")

print("\nPossible Weak Picks:")
print("- Some pages have low traffic because they are new.")
print("- Seasonal pages may naturally receive fewer impressions.")
print("- Some pages may already be optimized.")

print("\nLeakage Check:")
print("✓ No future window used")
print("✓ No label-derived features used")
print("✓ Only current SEO signals were used")


Weak Picks + Leakage Check

Possible Weak Picks:
- Some pages have low traffic because they are new.
- Seasonal pages may naturally receive fewer impressions.
- Some pages may already be optimized.

Leakage Check:
✓ No future window used
✓ No label-derived features used
✓ Only current SEO signals were used


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.